In [ ]:
import json 
import os 
from tqdm import tqdm 

nlpeer_path = "venues_data/ARR_EMNLP-24/data"
complete_reviews = []

for folders in tqdm(os.listdir(nlpeer_path)):
    if folders == 'meta.json':
        continue

    v1_folder = os.path.join(nlpeer_path, folders, 'v1', 'reviews.json')

    with open(v1_folder, 'r') as f:
        review_data = json.load(f)
    
    if len(review_data) != 0:
        review_data = review_data[0]
        complete_reviews.append({'id': folders, 'review_id': review_data['rid'], 'summary': review_data['report']['paper_summary'],
                                'weaknesses': review_data['report']['summary_of_weaknesses'], 'suggestions': review_data['report']['comments_suggestions_and_typos']})

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv

key_path = 'keys.env'
load_dotenv(dotenv_path=key_path) 
api_key = os.getenv("API_KEY")
client = OpenAI(api_key=api_key)

def ask_gpt(prompt):
    completion = client.chat.completions.create(
        model="gpt-5",
        messages=[
            {"role": "assistant", "content": "You are a helpful assistant."},
            {
                "role": "user",
                "content": prompt
            }
        ],
        reasoning_effort="minimal",
    )

    return completion.choices[0].message.content

In [ ]:
from tqdm import tqdm 
import re 

runs = 3
for i in range(runs):
    for reviews in tqdm(complete_reviews, desc=f"Run {i+1}/{runs}"):
        weaknesses = reviews['weaknesses']
        suggestions = reviews['suggestions']

        prompt = f""" 
        Provided to you is a review comments on the weaknesses and suggestions for the paper. Please classify it as either relevant to the introduction section, related works section, or neither. 
        Valid classifications are either: Introduction, Related Works, Neither.

        Please have your output strictly at the end of your response and in this format:
        {'classification: ...'}

        Weaknesses: 
        {weaknesses}
        Suggestions:
        {suggestions} """

        response = ask_gpt(prompt)
        match = re.search(r"classification:\s*(.*)", response, re.IGNORECASE | re.DOTALL)
        if match:
            reviews['gpt_classification'] = match.group(1)
        else:
            reviews['gpt_classification'] = None
    
    with open(f"processed_data/nlpeer_v2_classification_run_{i+1}.json", 'w') as f:
        json.dump(complete_reviews, f, indent=2)

In [ ]:
import json
from collections import Counter
from pathlib import Path

def load_ids_from_json(path):
    """Load a JSON file assumed to be a list of dicts with an 'id' key."""
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    items = []
    for item in data:
        # guard in case some items don't have 'id'
        if isinstance(item, dict) and "id" in item:
            items.append({'id': item["id"], 'gpt_classification': item['gpt_classification']})
    return items


def most_common_ids_across_files(file1, file2, file3, top_n=None):
    all_items = []
    for path in [file1, file2, file3]:
        all_items.extend(load_ids_from_json(path))
    
    id_counts = Counter(item["id"] for item in all_items)

    id_to_class = {}
    for item in all_items:
        _id = item["id"]
        if _id not in id_to_class:
            id_to_class[_id] = item["gpt_classification"]

    common = id_counts.most_common(top_n) if top_n is not None else id_counts.most_common()

    result = [
        {
            "id": _id,
            "count": count,
            "gpt_classification": id_to_class.get(_id)
        }
        for _id, count in common
    ]

    return result


file1 = "processed_data/nlpeer_v2_classification_run_1.json"
file2 = "processed_data/nlpeer_v2_classification_run_2.json"
file3 = "processed_data/nlpeer_v2_classification_run_3.json"

common_ids = most_common_ids_across_files(file1, file2, file3)

with open("processed_data/common_ids_across_runs_v2.json", "w", encoding="utf-8") as f:
    json.dump(common_ids, f, indent=2)

print(len(common_ids))

In [ ]:
rw_intro = []

for review in common_ids:
    if review['gpt_classification'] != "Neither":
        rw_intro.append(review)

In [ ]:
import shutil
from pathlib import Path
from tqdm import tqdm 

def copy_file_to_folder(src_file, dst_folder, filename):
    src_file = Path(src_file)
    dst_folder = Path(dst_folder)

    dst_folder.mkdir(parents=True, exist_ok=True)  # create folder if it doesn't exist

    shutil.copy2(src_file, dst_folder / f"{filename}.tei")  # preserves metadata

for review in tqdm(rw_intro):
    folder = review['id']
    try:
        copy_file_to_folder(f"venues_data/ARR_EMNLP-24/data/{folder}/v1/paper.tei", "processed_data/tei_files", f"pdf_{folder[10:]}")
    except FileNotFoundError:
        print(f"Paper {folder[10:]} does not have a paper.tei file")

In [ ]:
from lxml import etree
from lxml.etree import _Element
import re
from typing import List, Tuple, Optional, Dict

"""
Extract Introduction and Related-Work-equivalent sections from ARR/ACL-style
papers that have already been converted to TEI XML (e.g. via GROBID).

Fixes applied to the original snippet:
- Added the missing `TEI_NS` namespace map (required by build_ordered_blocks_from_tei).
- Added the missing `TOP_LEVEL_HEADING_RE` regex (required by the paragraph fallback).
- Removed `find_by_id`, which referenced an undefined external `items` list / key
  ("gpt_classification") unrelated to section extraction — dead code that could
  never run correctly here.
- Generalized "special" synonym matching so it covers BOTH Introduction and
  Related Work equivalents, not just Related Work. ARR papers phrase these
  headings inconsistently (e.g. "Prior Work", "Literature Review", "Background
  and Related Work"), so both need fuzzy matching, not just exact substrings.
- Added `extract_intro_and_related_work()`, a single entry point that returns
  both sections in one call.
"""

# --- TEI namespace (GROBID output uses this) -------------------------------
TEI_NS = {"tei": "http://www.tei-c.org/ns/1.0"}

# --- Canonical section categories and their common ARR/ACL-style synonyms --
SECTION_EQUIVALENTS: Dict[str, set] = {
    "introduction": {
        "introduction",
        "intro",
    },
    "related work": {
        "related work",
        "related works",
        "background",
        "prior work",
        "previous work",
        "literature review",
        "background and related work",
        "related work and background",
        "past",
        "current",
    },
}

# Matches a heading like "1 Introduction", "1. Introduction", "2.1 Related Work"
TOP_LEVEL_HEADING_RE = re.compile(r"^\s*(\d+)\.?\s+(.*)$")

FIG_TABLE_RE = re.compile(r"\b(?:figure|table)\s*\d+\s*:", flags=re.IGNORECASE)
FIG_RE = re.compile(r"\bfigure\s*\d+\s*:", flags=re.IGNORECASE)


def normalize_target(s: str) -> str:
    s = (s or "").strip()
    # drop any leading numbering like "2 " or "2. " or "2.1 "
    s = re.sub(r"^\d+(\.\d+)*\s+", "", s)
    return s.lower()


def local_name(el: _Element) -> str:
    return etree.QName(el).localname


def get_text(el) -> str:
    if el is None:
        return ""
    return "".join(el.itertext()).strip()


def is_special_target(norm: str) -> Optional[str]:
    """
    Returns the canonical category name ("introduction" / "related work")
    if norm matches one of the recognized categories, else None.
    """
    for canonical, synonyms in SECTION_EQUIVALENTS.items():
        if norm in synonyms or norm == canonical:
            return canonical
    return None


def candidate_matches_special(candidate_title: str, canonical: str) -> bool:
    cand = (candidate_title or "").lower()
    for tok in SECTION_EQUIVALENTS.get(canonical, set()):
        if tok in cand:
            return True
    return False


def matches_candidate(candidate_title: str, target_norm: str) -> bool:
    """
    Return True if candidate_title matches target_norm according to rules.
    Defensive: if target_norm is empty, do NOT match anything.
    """
    if not target_norm or not candidate_title:
        return False
    cand_low = candidate_title.lower()

    canonical = is_special_target(target_norm)
    if canonical is not None:
        return candidate_matches_special(cand_low, canonical)

    # otherwise plain substring match either direction
    return (target_norm in cand_low) or (cand_low in target_norm)


def find_top_level_heading_blocks_using_title(blocks: List[dict]) -> List[Tuple[Optional[int], str, int]]:
    """
    Build a list of top-level heading candidates from <head> blocks.

    Returns list of tuples (section_number_or_None, heading_text, block_index).
    Matching is done against heading_text, NOT the @n value; @n is only used
    to tell top-level headings apart from dotted subsections (e.g. "3.1").
    """
    headings: List[Tuple[Optional[int], str, int]] = []

    for idx, b in enumerate(blocks):
        if b["type"] != "head":
            continue
        el = b.get("element")
        head_text = b["text"].strip()
        n_attr = None
        if el is not None:
            n_attr = el.get("n")
            if n_attr:
                n_attr = n_attr.strip() or None

        # Integer n -> top-level, record the number
        if n_attr and re.fullmatch(r"\d+", n_attr):
            headings.append((int(n_attr), head_text, idx))
            continue

        # Dotted n (e.g. "3.1") -> subsection, skip as a top-level boundary
        if n_attr and re.fullmatch(r"\d+\.\d+(\.\d+)*", n_attr):
            continue

        # No usable n -> still include as an unnumbered top-level candidate,
        # since matching happens on head text, not on n.
        headings.append((None, head_text, idx))

    return headings


def fallback_find_headings_in_paragraphs_using_title(blocks: List[dict]) -> List[Tuple[Optional[int], str, int]]:
    """
    Fallback heading detection: scans paragraph blocks for leading numbered
    headings OR short title-case paragraphs that look like headings.
    """
    headings = []
    for idx, b in enumerate(blocks):
        if b["type"] != "p":
            continue
        text = b["text"].strip()
        m = TOP_LEVEL_HEADING_RE.match(text)
        if m:
            num = int(m.group(1))
            title = m.group(2).strip().rstrip(" .:;,-")
            headings.append((num, title, idx))
        else:
            words = text.split()
            if 1 <= len(words) <= 6 and text == text.title():
                headings.append((None, text.rstrip(" .:;,-"), idx))
    return headings


def build_ordered_blocks_from_tei(tei_path: str) -> Tuple[List[dict], str]:
    """
    Parse TEI and build a list of blocks in document order.
    Each block: {"type": "head"|"p", "text": str, "element": Element}
    """
    tree = etree.parse(tei_path)
    root = tree.getroot()

    body = root.find(".//tei:text/tei:body", namespaces=TEI_NS)
    source = body if body is not None else root

    blocks: List[dict] = []
    for el in source.iter():
        name = local_name(el)
        if name == "head":
            text = get_text(el)
            if text:
                blocks.append({"type": "head", "text": text, "element": el})
        elif name in ("p", "ab"):
            text = get_text(el)
            if text:
                blocks.append({"type": "p", "text": text, "element": el})

    preview_pieces = [b["text"] for b in blocks]
    full_text_preview = " \n\n ".join(preview_pieces)
    return blocks, full_text_preview


def debug_blocks(blocks: List[dict], start: int, end: int):
    print(f"DEBUG: slicing blocks [{start} .. {end - 1}] (inclusive)")
    for i in range(start, end):
        b = blocks[i]
        txt = b["text"].strip().replace("\n", "\\n")
        print(f"  idx={i:3} type={b['type']:4} text_preview={txt[:120]!r}")


def extract_section_in_tei(
    tei_path: str,
    section: str,
    *,
    blocks_cache: Optional[List[dict]] = None,
    debug: bool = False,
) -> Dict[str, Optional[object]]:
    """
    Match on <head> text (not @n), slice blocks between the matched top-level
    <head> and the next top-level <head>, exclude Figure/Table caption
    paragraphs, and flag whether any "Figure <n>:" occurred in the slice.

    `section` should be a canonical name ("introduction" or "related work")
    or any of their recognized synonyms, or a raw heading string.
    """
    empty_result = {
        "matched_heading": None, "content": None,
        "start_block_idx": None, "end_block_idx": None,
        "included_blocks": [], "excluded_blocks": [],
        "has_figure": False,
    }

    if section is None:
        return empty_result

    target_norm = normalize_target(section)
    if not target_norm:
        return empty_result

    blocks = blocks_cache if blocks_cache is not None else build_ordered_blocks_from_tei(tei_path)[0]

    headings = find_top_level_heading_blocks_using_title(blocks)
    if not headings:
        headings = fallback_find_headings_in_paragraphs_using_title(blocks)

    if debug:
        print("DEBUG: discovered headings (num_or_None, title, block_idx):")
        for h in headings:
            print("  ", h)

    if not headings:
        return empty_result

    matched_idx = None
    matched_title = None
    for hi, (num, title, block_idx) in enumerate(headings):
        if matches_candidate(title, target_norm):
            matched_idx = hi
            matched_title = title
            break

    # fallback: try matching by a leading numeric prefix in the provided string
    if matched_idx is None:
        mnum = re.match(r"^\s*(\d+)\s+", section)
        if mnum:
            wanted = int(mnum.group(1))
            for hi, (num, title, block_idx) in enumerate(headings):
                if num == wanted:
                    matched_idx = hi
                    matched_title = title
                    break

    if matched_idx is None:
        if debug:
            print("DEBUG: no matching heading found for target:", repr(target_norm))
        return empty_result

    start_block_idx = headings[matched_idx][2]
    end_block_idx = headings[matched_idx + 1][2] if matched_idx + 1 < len(headings) else len(blocks)

    if debug:
        print(f"DEBUG: matched heading: {matched_title!r} at headings index {matched_idx}, "
              f"start_block_idx={start_block_idx}, end_block_idx={end_block_idx}")
        debug_blocks(blocks, start_block_idx, end_block_idx)

    included_blocks: List[int] = []
    excluded_blocks: List[int] = []
    pieces: List[str] = []
    has_figure = False

    for bi in range(start_block_idx, end_block_idx):
        b = blocks[bi]
        text = b["text"] or ""
        if FIG_RE.search(text):
            has_figure = True

        if b["type"] == "p" and FIG_TABLE_RE.search(text):
            excluded_blocks.append(bi)
            if debug:
                print(f"DEBUG: excluding block idx={bi} (figure/table paragraph): {b['text'][:60]!r}")
            continue

        included_blocks.append(bi)
        pieces.append(text.strip())

    content = "\n\n".join(pieces).strip() or None

    if debug:
        print("DEBUG: included_blocks:", included_blocks)
        print("DEBUG: excluded_blocks:", excluded_blocks)
        print("DEBUG: content length:", 0 if content is None else len(content))
        print("DEBUG: has_figure:", has_figure)
        if content:
            print("DEBUG: content preview:", content[:400])

    return {
        "matched_heading": matched_title,
        "content": content,
        "start_block_idx": start_block_idx,
        "end_block_idx": end_block_idx,
        "included_blocks": included_blocks,
        "excluded_blocks": excluded_blocks,
        "has_figure": has_figure,
    }


def extract_intro_and_related_work(tei_path: str, *, debug: bool = False) -> Dict[str, Dict[str, Optional[object]]]:
    """
    Convenience entry point: extracts both the Introduction-equivalent and
    Related-Work-equivalent sections from a single TEI file in one parse.
    """
    blocks, _ = build_ordered_blocks_from_tei(tei_path)
    return {
        "introduction": extract_section_in_tei(tei_path, "introduction", blocks_cache=blocks, debug=debug),
        "related_work": extract_section_in_tei(tei_path, "related work", blocks_cache=blocks, debug=debug),
    }

In [ ]:
from lxml import etree
from typing import List, Dict, Optional, Union
import re

TEI_NS = {"tei": "http://www.tei-c.org/ns/1.0"}

# -------------------------
# Extraction of biblStructs in tei files
# -------------------------
def _get_text(el: Optional[etree._Element]) -> str:
    return "" if el is None else "".join(el.itertext()).strip()

def extract_bibl_structs(tei_path: str) -> List[Dict]:
    """
    Parse TEI file at tei_path and return a filtered list of dicts, one per <biblStruct>.
    Filtering heuristic:
      - include entries that have a monogr title (journal/book-like entries), OR
      - include analytic-title entries only when they also have some useful metadata
        (year, pages, or a useful idno like DOI/arXiv).
      - exclude entries that are essentially empty (e.g., only MD5 idno, empty imprint/date).
    """
    tree = etree.parse(tei_path)
    root = tree.getroot()
    bibl_nodes = root.xpath(".//tei:biblStruct", namespaces=TEI_NS)

    out = []
    for b in bibl_nodes:
        # quick introspection: detect presence of analytic/monogr titles
        analytic_title_nodes = b.xpath(".//tei:analytic/tei:title", namespaces=TEI_NS)
        monogr_title_nodes = b.xpath(".//tei:monogr/tei:title", namespaces=TEI_NS)

        # preliminary metadata checks: date, pages, idnos
        # Year detection
        year = ""
        date_nodes = b.xpath(".//tei:imprint/tei:date", namespaces=TEI_NS)
        if date_nodes:
            d = date_nodes[0]
            when = d.get("when")
            if when and re.search(r"\d{4}", when):
                year = re.search(r"(\d{4})", when).group(1)
            else:
                dt = _get_text(d)
                m = re.search(r"(\d{4})", dt)
                if m:
                    year = m.group(1)

        # Pages detection
        page_nodes = b.xpath(".//tei:imprint/tei:biblScope[@unit='page']", namespaces=TEI_NS)
        pages = ""
        if page_nodes:
            el = page_nodes[0]
            frm = el.get("from")
            to = el.get("to")
            if frm and to:
                pages = f"{frm}--{to}"
            else:
                pages = _get_text(el)

        # idnos: gather and check for useful identifiers
        idno_nodes = b.xpath(".//tei:idno", namespaces=TEI_NS)
        idnos = {}
        for idn in idno_nodes:
            typ = (idn.get("type") or "").strip()
            val = _get_text(idn)
            if typ:
                idnos[typ] = val
            else:
                if re.search(r"10\.\d{4,9}/", val):
                    idnos["DOI"] = val
                elif re.search(r"arxiv", val, flags=re.I) or re.search(r"\d{4}\.\d{4,}", val):
                    idnos["arXiv"] = val
                else:
                    idnos.setdefault("other", []).append(val)

        has_useful_id = bool(idnos.get("DOI") or idnos.get("arXiv"))
        has_year_or_pages = bool(year or pages)

        # Decide whether to include this biblStruct
        include = False
        if monogr_title_nodes:
            # monogr entries are usually meaningful (journal/book)
            include = True
        else:
            # if only analytic title exists, require at least one useful metadata field
            if analytic_title_nodes and (has_year_or_pages or has_useful_id):
                include = True

        if not include:
            # skip this biblStruct as it appears to be non-informative (e.g., only MD5 id)
            continue

        # Build the item dict (same structure as before)
        item = {
            "xml_id": b.get("{http://www.w3.org/XML/1998/namespace}id") or b.get("xml:id") or b.get("id"),
            "raw_xml": etree.tostring(b, encoding="unicode", with_tail=False),
            "title": "",
            "authors": [],  # list of {"surname":..., "forenames":[...], "raw":...}
            "venue": "",
            "publisher": "",
            "year": year,
            "pages": pages,
            "idnos": idnos
        }

        # Title: prefer analytic level='a' then analytic then monogr title then generic
        title_nodes = b.xpath(".//tei:analytic/tei:title[@level='a']|.//tei:analytic/tei:title|.//tei:monogr/tei:title|.//tei:title", namespaces=TEI_NS)
        if title_nodes:
            item["title"] = _get_text(title_nodes[0])

        # Authors: try extracting persName children
        authors = []
        for au in b.xpath(".//tei:author", namespaces=TEI_NS):
            pers = au.find(".//{http://www.tei-c.org/ns/1.0}persName")
            if pers is not None:
                surname_el = pers.find(".//{http://www.tei-c.org/ns/1.0}surname")
                forename_els = pers.findall(".//{http://www.tei-c.org/ns/1.0}forename")
                surname = _get_text(surname_el) if surname_el is not None else ""
                forenames = [_get_text(fn) for fn in forename_els if _get_text(fn)]
                raw = _get_text(pers)
                authors.append({"surname": surname, "forenames": forenames, "raw": raw})
            else:
                # fallback: use author text
                at = _get_text(au)
                if at:
                    parts = at.split()
                    if len(parts) >= 2:
                        surname = parts[-1]
                        forenames = parts[:-1]
                    else:
                        surname = at
                        forenames = []
                    authors.append({"surname": surname, "forenames": forenames, "raw": at})
        item["authors"] = authors

        # Venue / monogr title (prefer monogr/journal)
        journal = b.xpath(".//tei:monogr/tei:title[@level='j']", namespaces=TEI_NS)
        monogr = b.xpath(".//tei:monogr/tei:title", namespaces=TEI_NS)
        if journal:
            item["venue"] = _get_text(journal[0])
        elif monogr:
            item["venue"] = _get_text(monogr[0])
        else:
            gen = b.xpath(".//tei:title", namespaces=TEI_NS)
            item["venue"] = _get_text(gen[0]) if gen else ""

        # Publisher (if present)
        pub = b.xpath(".//tei:imprint/tei:publisher", namespaces=TEI_NS)
        if pub:
            item["publisher"] = _get_text(pub[0])

        # Keep idnos (already collected)
        item["idnos"] = idnos

        out.append(item)

    return out

# -------------------------
# ACL formatting utilities
# -------------------------
def _format_author_acl(author: Dict) -> str:
    """
    Turn {"surname": "...", "forenames": [...]} into "Surname, F." (use first forename initial).
    If multiple forenames, use initials for each (e.g., "John Paul" -> "J.P.").
    If surname missing, use raw.
    """
    surname = author.get("surname") or ""
    forenames = author.get("forenames") or []
    raw = author.get("raw") or ""
    if not surname and not forenames:
        return raw or ""
    # build initials from forenames (use first forename only or all? we use all for initials)
    initials = ""
    if forenames:
        initials = " ".join(
            "".join([part[0] + "." for part in fn.split()]) for fn in forenames
        )
        # collapse spaces to single space
        initials = re.sub(r"\s+", " ", initials).strip()
    if surname and initials:
        return f"{surname}, {initials}"
    if surname:
        return surname
    return raw

def _join_authors_acl(authors: List[Dict]) -> str:
    """
    Join formatted authors in ACL style:
      - 1 author: A.
      - 2 authors: A. and B.
      - >2: A., B., and C.
    """
    formatted = [_format_author_acl(a) for a in authors if _format_author_acl(a)]
    if not formatted:
        return ""
    if len(formatted) == 1:
        return formatted[0]
    if len(formatted) == 2:
        return f"{formatted[0]} and {formatted[1]}"
    # 3+ authors
    return ", ".join(formatted[:-1]) + ", and " + formatted[-1]

def format_reference_acl(ref: Dict) -> str:
    """
    Format a single reference dict (from extract_bibl_structs) into ACL-like citation string.
    """
    authors = _join_authors_acl(ref.get("authors", []))
    year = ref.get("year") or ""
    title = ref.get("title") or ""
    venue = ref.get("venue") or ""
    pages = ref.get("pages") or ""
    doi = None
    idnos = ref.get("idnos", {})
    # prefer DOI, then arXiv, then any other id
    if idnos:
        doi = idnos.get("DOI") or idnos.get("doi") or idnos.get("arXiv")
        if not doi:
            # if 'other' is list, pick first
            oth = idnos.get("other")
            if isinstance(oth, list) and oth:
                doi = oth[0]

    parts = []
    if authors:
        parts.append(authors)
    if year:
        parts.append(year + ".")
    if title:
        parts.append(title.rstrip(".") + ".")
    if venue:
        parts.append("In " + venue.rstrip(".") + ".")
    if pages:
        parts.append("pp. " + pages)
    if doi:
        parts.append(doi)
    # Join with space
    return " ".join(p for p in parts if p).strip()

In [ ]:
from tqdm import tqdm
import os

total_text = []
tei_folder = 'processed_data/tei_files'
failed_to_process = []  # list of (file_id, section_name) tuples

for file in tqdm(os.listdir(tei_folder)):
    if '.tei' not in file:
        continue

    file_id = (file[4:]).split('.tei')[0]
    tei_path = os.path.join(tei_folder, f"pdf_{file_id}.tei")

    try:
        result = extract_intro_and_related_work(tei_path, debug=False)
    except Exception as e:
        print(f"Error parsing {tei_path}: {e}")
        failed_to_process.append((file_id, "parse_error"))
        continue

    for section_name, tei_section_text in result.items():
        if tei_section_text["matched_heading"] is None:
            failed_to_process.append((file_id, section_name))
            continue

        total_text.append({
            'id': file,
            'file_id': file_id,
            'section': section_name,
            'matched_heading': tei_section_text['matched_heading'],
            'text': tei_section_text['content'],
            'has_figure': tei_section_text['has_figure'],
        })


In [ ]:
import json 

with open("complete_dataset_for_annotation.json", 'w') as f:
    json.dump(total_text, f, indent=2)

In [ ]:
import os
from pathlib import Path

skipped_due_to_figures = []

def save_to_txt(dict_list, folder_path, base_filename="item"):
    print("Number of items in dict_list:", len(dict_list))

    folder = Path(folder_path)
    folder.mkdir(parents=True, exist_ok=True)
    print("Saving files into folder:", folder.resolve())

    saved_count = 0
    for i, item in enumerate(dict_list, start=1):
        # Skip items that contain a figure — don't write them at all
        if item.get("has_figure"):
            skipped_due_to_figures.append(item.get("id", i))
            continue

        # Only use the 'text' field
        text = item.get("text", "")

        # Use 'id' in the filename if present, else fallback to the index
        file_id = item.get("id", i)
        file_path = folder / f"{base_filename}_{i}.txt"  # or use file_id if you prefer

        with open(file_path, "w", encoding="utf-8") as f:
            f.write(str(text))

        saved_count += 1

    print(f"Saved {saved_count} files, skipped {len(skipped_due_to_figures)} due to figures")


save_to_txt(total_text, "to_annotate/", "nlpeer_v2")
print(f"These files were skipped because they have figures: {skipped_due_to_figures}")